In [1]:
import mysql.connector
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

def get_product_recommendations(product_id):
    # 1. فتح اتصال بقاعدة البيانات
    connection = mysql.connector.connect(
        host='localhost',
        user='root',
        password='1234',
        database='wp-ecommerce'
    )

    # 2. جلب اسم المنتج بناءً على المعرّف
    cursor = connection.cursor(dictionary=True)
    sql = 'SELECT post_title FROM wp_posts WHERE ID=(%s)'
    cursor.execute(sql, (product_id,))
    result = cursor.fetchone()
    product_name = result['post_title'] if result else 'Unknown Product'

    # 3. جمع الطلبات التي تحتوي على أكثر من منتج
    cursor.execute('SELECT order_id FROM wp_wc_order_stats ORDER BY order_id')
    orders = cursor.fetchall()
    product_lists = []
    for order in orders:
        order_id = order['order_id']
        sql = 'SELECT product_id FROM wp_wc_order_product_lookup WHERE order_id=(%s)'
        cursor.execute(sql, (order_id,))
        products = cursor.fetchall()
        product_ids = [p['product_id'] for p in products if p['product_id'] > 0]
        if len(product_ids) > 1:
            product_lists.append(product_ids)
    cursor.close()

    # 4. تحويل البيانات إلى صيغة تحليل
    from mlxtend.preprocessing import TransactionEncoder
    te = TransactionEncoder()
    te_model = te.fit(product_lists)
    rows = te_model.transform(product_lists)
    df_transactions = pd.DataFrame(rows, columns=te_model.columns_)

    # 5. إنشاء قواعد الارتباط
    frequent_itemsets = apriori(df_transactions, min_support=0.001, use_colnames=True)
    rules = association_rules(frequent_itemsets, metric='confidence', min_threshold=0.001)
    rules = rules.sort_values('confidence', ascending=False)

    # 6. جلب التوصيات للمنتج المحدد
    recommendations = rules[rules['antecedents'].apply(lambda x: product_id in x)]
    recommendations = recommendations[['consequents', 'confidence']]
    recommendations = recommendations.sort_values('confidence', ascending=False).head(6)

    # 7. تحويل التوصيات إلى جدول مع أسماء المنتجات
    result_df = pd.DataFrame(columns=['product_id', 'post_title', 'confidence'])
    for _, row in recommendations.iterrows():
        for prod_id in row['consequents']:
            cursor = connection.cursor(dictionary=True)
            sql = 'SELECT post_title FROM wp_posts WHERE ID=(%s)'
            cursor.execute(sql, (int(prod_id),))
            result = cursor.fetchone()
            name = result['post_title'] if result else 'Unknown Product'
            obj = {
                'product_id': int(prod_id),
                'post_title': name,
                'confidence': round(row['confidence'] * 100, 2)
            }
            result_df = pd.concat([result_df, pd.DataFrame([obj])], ignore_index=True)
            cursor.close()

    # 8. إغلاق الاتصال
    connection.close()

    # 9. طباعة النتائج
    print(f'Recommendations for {product_id} ({product_name}):')
    print(result_df)

    return result_df

# تشغيل الدالة لمنتج معين
get_product_recommendations(56306)

Recommendations for 56306 (مجموعة العناية بالبشرة-BLVD 13):
  product_id                   post_title  confidence
0      56350  شامبو مغزي للشعر-Boho Bunny       75.00
1      56358    زيت تصفيف للشعر-URBN Glow       60.00
2      56323             ميكاب-Wild Child       22.22
3      56350  شامبو مغزي للشعر-Boho Bunny       18.52
4      56347                  ميكاب-SHADE       14.81
5      56358    زيت تصفيف للشعر-URBN Glow       14.81


C:\Users\Aiham JUMAH\AppData\Local\Temp\ipykernel_34280\541613753.py:66: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result_df = pd.concat([result_df, pd.DataFrame([obj])], ignore_index=True)


,product_id,post_title,confidence
0,56350,شامبو مغزي للشعر-Boho Bunny,75.00
1,56358,زيت تصفيف للشعر-URBN Glow,60.00
2,56323,ميكاب-Wild Child,22.22
3,56350,شامبو مغزي للشعر-Boho Bunny,18.52
4,56347,ميكاب-SHADE,14.81
5,56358,زيت تصفيف للشعر-URBN Glow,14.81
